In [1]:
# ============================================================
# CELL 1 — HOM simulator tools (QuTiP), NO LOSS baseline
# Stable internal scenario IDs + separate display labels in Cell 2.
#
# Internal scenario IDs:
#   fock11, superpos01, coherent, smsv
#
# Output keys are ALWAYS scenario IDs (never relabeled).
# ============================================================

import numpy as np
import math
from qutip import basis, ket2dm, tensor, qeye, destroy, coherent, squeeze

EPS = 1e-15

# ---------------- Conversions ----------------
def dB_to_r(dB: float) -> float:
    """variance squeezing dB: dB = 10 log10(e^{2r}) -> r = (dB/10)*(ln10/2)"""
    return (dB / 10.0) * (np.log(10.0) / 2.0)

# ---------------- Gerry–Knight beamsplitter ----------------
def gk_beamsplitter_unitary(N: int, theta: float, phi: float):
    """Gerry–Knight BS, balanced at theta=pi/2."""
    a = tensor(destroy(N), qeye(N))
    b = tensor(qeye(N), destroy(N))
    G = np.exp(1j*phi) * a.dag()*b - np.exp(-1j*phi) * a*b.dag()
    return ((theta/2.0) * G).expm()

# ---------------- Metrics (2-mode) ----------------
def _real(x):
    return float(np.real_if_close(x, tol=1e6))

def coincidence_probability_onoff(rho2, N: int):
    """On/off coincidence Pc = 1 - P0_1 - P0_2 + P0_both."""
    P0 = ket2dm(basis(N, 0))
    I = qeye(N)
    P0_1 = tensor(P0, I)
    P0_2 = tensor(I, P0)
    P0_b = tensor(P0, P0)
    p0_1 = _real((P0_1 * rho2).tr())
    p0_2 = _real((P0_2 * rho2).tr())
    p0_b = _real((P0_b * rho2).tr())
    Pc = 1.0 - p0_1 - p0_2 + p0_b
    return max(0.0, min(1.0, Pc))

def g2_12(rho2, N: int):
    a = tensor(destroy(N), qeye(N))
    b = tensor(qeye(N), destroy(N))
    n1 = a.dag()*a
    n2 = b.dag()*b
    n1m = _real((n1 * rho2).tr())
    n2m = _real((n2 * rho2).tr())
    n12 = _real(((n1*n2) * rho2).tr())
    denom = n1m * n2m
    if denom < 1e-14:
        return np.nan
    return n12 / denom

def nrf(rho2, N: int):
    """NRF = Var(n1-n2)/<n1+n2>."""
    a = tensor(destroy(N), qeye(N))
    b = tensor(qeye(N), destroy(N))
    n1 = a.dag()*a
    n2 = b.dag()*b
    n1m = _real((n1 * rho2).tr())
    n2m = _real((n2 * rho2).tr())
    nd = (n1 - n2)
    ndm = _real((nd * rho2).tr())
    nd2 = _real(((nd*nd) * rho2).tr())
    var = nd2 - ndm**2
    denom = n1m + n2m
    if denom < 1e-14:
        return np.nan
    return var / denom

# ---------------- Scenario registry (IDs) ----------------
SCENARIOS = {
    "fock11": {
        "label": "Fock |1⟩⊗|1⟩",
    },
    "superpos01": {
        "label": "Superpos (|0⟩+|1⟩)/√2 ⊗ same",
    },
    "coherent": {
        "label": "Coherent α ⊗ α",
    },
    "smsv": {
        "label": "SMSV ⊗ SMSV",
    },
}

SCENARIO_IDS = list(SCENARIOS.keys())

def build_input_by_id(sid: str, N: int,
                      alpha_abs=1.0, alpha_phase=0.0,
                      r_dB=6.0, phi_sq1=0.0, phi_sq2=np.pi/2):
    if sid == "fock11":
        return ket2dm(tensor(basis(N, 1), basis(N, 1)))

    if sid == "superpos01":
        psi1 = (basis(N, 0) + basis(N, 1)).unit()
        return ket2dm(tensor(psi1, psi1))

    if sid == "coherent":
        alpha = alpha_abs * np.exp(1j * alpha_phase)
        return ket2dm(tensor(coherent(N, alpha), coherent(N, alpha)))

    if sid == "smsv":
        r = dB_to_r(r_dB)
        Sa = squeeze(N, r * np.exp(1j*phi_sq1))
        Sb = squeeze(N, r * np.exp(1j*phi_sq2))
        psi = tensor(Sa * basis(N, 0), Sb * basis(N, 0))
        return ket2dm(psi)

    raise ValueError(f"Unknown scenario id: {sid}")

# ---------------- Sweep runner (no loss) ----------------
def run_hom_sweep(
    scenario_ids,
    N=12,
    theta_pts=61,
    phi_bs=np.pi/2,
    alpha_abs=1.0,
    alpha_phase=0.0,
    r_dB=6.0,
    phi_sq1=0.0,
    phi_sq2=np.pi/2,
):
    theta_list = np.linspace(0.0, np.pi, int(theta_pts))
    results = {}

    for sid in scenario_ids:
        rho_in = build_input_by_id(
            sid, int(N),
            alpha_abs=alpha_abs, alpha_phase=alpha_phase,
            r_dB=r_dB, phi_sq1=phi_sq1, phi_sq2=phi_sq2
        )

        Pc_list, g2_list, NRF_list = [], [], []
        for theta in theta_list:
            U = gk_beamsplitter_unitary(int(N), float(theta), float(phi_bs))
            rho_out = U * rho_in * U.dag()
            Pc_list.append(coincidence_probability_onoff(rho_out, int(N)))
            g2_list.append(g2_12(rho_out, int(N)))
            NRF_list.append(nrf(rho_out, int(N)))

        results[sid] = {
            "Pc": np.array(Pc_list, dtype=float),
            "g2": np.array(g2_list, dtype=float),
            "NRF": np.array(NRF_list, dtype=float),
        }

    meta = {
        "N_cut": int(N),
        "theta_pts": int(theta_pts),
        "phi_bs": float(phi_bs),
        "alpha_abs": float(alpha_abs),
        "alpha_phase": float(alpha_phase),
        "r_dB": float(r_dB),
        "phi_sq1": float(phi_sq1),
        "phi_sq2": float(phi_sq2),
        "delta_phi_sq": float(phi_sq2 - phi_sq1),
    }
    return theta_list, results, meta

# ---------------- Engineering specs from curves (state-agnostic) ----------------
def _finite_min(arr):
    a = np.array(arr, dtype=float); m = np.isfinite(a)
    return float(np.min(a[m])) if np.any(m) else np.nan

def _finite_max(arr):
    a = np.array(arr, dtype=float); m = np.isfinite(a)
    return float(np.max(a[m])) if np.any(m) else np.nan

def _argmin(arr):
    a = np.array(arr, dtype=float); m = np.isfinite(a)
    if not np.any(m): return None
    return int(np.where(m)[0][np.argmin(a[m])])

def _tolerance_halfwidth(x, mask, idx_center):
    if idx_center < 0 or idx_center >= len(x) or not mask[idx_center]:
        return np.nan
    i = idx_center
    while i-1 >= 0 and mask[i-1]: i -= 1
    j = idx_center
    while j+1 < len(x) and mask[j+1]: j += 1
    return float(min(x[idx_center] - x[i], x[j] - x[idx_center]))

def compute_specs(theta_list, curve, delta_pc=0.02, delta_nrf=0.05):
    x = theta_list/np.pi
    Pc  = np.array(curve["Pc"], dtype=float)
    g2  = np.array(curve["g2"], dtype=float)
    NRF = np.array(curve["NRF"], dtype=float)

    idx_bal = int(np.argmin(np.abs(theta_list - (np.pi/2))))
    Pc_bal  = float(Pc[idx_bal])
    g2_bal  = float(g2[idx_bal]) if np.isfinite(g2[idx_bal]) else np.nan
    NRF_bal = float(NRF[idx_bal]) if np.isfinite(NRF[idx_bal]) else np.nan

    Pc_min = _finite_min(Pc)
    Pc_max = _finite_max(Pc)
    idx_pmin = _argmin(Pc)
    theta_pmin = float(theta_list[idx_pmin]) if idx_pmin is not None else np.nan

    V_Pc = np.nan if not (np.isfinite(Pc_max) and Pc_max > EPS) else float((Pc_max - Pc_min)/Pc_max)

    tol_pc = np.nan
    if np.isfinite(Pc_min):
        tol_pc = _tolerance_halfwidth(x, (Pc <= Pc_min + delta_pc) & np.isfinite(Pc), idx_bal)

    NRF_min = _finite_min(NRF)
    tol_nrf = np.nan
    if np.isfinite(NRF_min):
        tol_nrf = _tolerance_halfwidth(x, (NRF <= NRF_min + delta_nrf) & np.isfinite(NRF), idx_bal)

    return dict(Pc_min=Pc_min, Pc_max=Pc_max, V_Pc=V_Pc, theta_min=theta_pmin,
                Pc_bal=Pc_bal, g2_bal=g2_bal, NRF_bal=NRF_bal,
                NRF_min=NRF_min, tol_pc=tol_pc, tol_nrf=tol_nrf)

print("Cell 1 loaded (IDs + display labels separated).")


Cell 1 loaded (IDs + display labels separated).


In [2]:
# ============================================================
# CELL 2 — HOM simulator UI (no scrolling) — MATCHES IDs-based Cell 1
# - 3 aligned plots (Pc, g2_12, NRF) side-by-side
# - 4 spec cards underneath (uniform height, top-aligned)
# - Stores last run for Cell 3:
#     HOM_LAST_META, HOM_LAST_IDS, HOM_LAST_DEEMPH_ID
#
# Requires Cell 1:
#   SCENARIOS, SCENARIO_IDS
#   run_hom_sweep(...)
#   compute_specs(...)
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import clear_output

# Fixed-axis limits (tune as needed)
G2_MAX  = 10.0
NRF_MAX = 5.0

def make_display_label(sid: str, meta: dict) -> str:
    """Dynamic legend label for display only; conditioning stays on scenario IDs."""
    base = SCENARIOS[sid]["label"]
    if sid == "coherent":
        return f"{base} (|α|={meta['alpha_abs']:.2f})"
    if sid == "smsv":
        return f"{base} (r={meta['r_dB']:.1f} dB, Δφ={meta['delta_phi_sq']:+.2f})"
    return base

def plot_three_panel(theta_list, results_by_id, meta, deemph_id=None):
    x = theta_list / np.pi
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    for ax in axes:
        try:
            ax.set_box_aspect(1)
        except Exception:
            ax.set_aspect("equal", adjustable="box")
        ax.grid(alpha=0.3)
        ax.axvline(0.5, linestyle="--", linewidth=1)

    panels = [
        ("Pc",  r"$P_c$ (on/off coincidence)", "Coincidence", (0.0, 1.0)),
        ("g2",  r"$g^{(2)}_{12}$",            "Cross-correlation", (0.0, G2_MAX)),
        ("NRF", "NRF",                         "Noise-reduction factor", (0.0, NRF_MAX)),
    ]

    for sid, res in results_by_id.items():
        label = make_display_label(sid, meta)

        # emphasis style (same as Cell 3)
        style = dict(lw=2, zorder=3)
        if deemph_id is not None and sid == deemph_id:
            style = dict(lw=4, ls="--", alpha=0.75, zorder=6)

        for ax, (ykey, ylabel, title, ylim) in zip(axes, panels):
            y = np.array(res[ykey], dtype=float)
            m = np.isfinite(y)
            if np.any(m):
                ax.plot(x[m], y[m], label=label, **style)
            ax.set_xlabel(r"$\theta/\pi$")
            ax.set_ylabel(ylabel)
            ax.set_title(title)
            ax.set_ylim(*ylim)

    axes[-1].legend(fontsize=7, loc="upper right")
    fig.tight_layout()
    plt.show()

def make_spec_card_html(label, s, delta_pc, delta_nrf):
    def f(x, nd=4):
        return "nan" if not np.isfinite(x) else f"{x:.{nd}f}"

    return f"""
    <div style="
        border:1px solid #999; border-radius:8px;
        padding:8px; width:187px;
        min-height:285px;
        box-sizing:border-box;
        font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace;
        font-size:11px; line-height:1.20;">
      <div style="font-weight:700; margin-bottom:6px;">{label}</div>

      <div><b>Pc contrast</b></div>
      <div>Pc_min = {f(s['Pc_min'])}</div>
      <div>Pc_max = {f(s['Pc_max'])}</div>
      <div>V_Pc   = {f(s['V_Pc'])}</div>
      <div>θ_min  = {f(s['theta_min'], nd=3)}</div>

      <div style="margin-top:6px;"><b>At θ=π/2</b></div>
      <div>Pc  = {f(s['Pc_bal'])}</div>
      <div>g2  = {f(s['g2_bal'])}</div>
      <div>NRF = {f(s['NRF_bal'])}</div>

      <div style="margin-top:6px;"><b>Diagnostics</b></div>
      <div>NRF_min = {f(s['NRF_min'])}</div>

      <div style="margin-top:6px;"><b>Tolerances</b></div>
      <div>tol_pc  = {f(s['tol_pc'])} (δPc={delta_pc:.3f})</div>
      <div>tol_nrf = {f(s['tol_nrf'])} (δNRF={delta_nrf:.3f})</div>
    </div>
    """

def make_hom_ui_no_scroll_cards():
    # Controls
    N_w = widgets.IntSlider(value=12, min=6, max=30, step=1, description="N_cut")
    theta_pts_w = widgets.IntSlider(value=61, min=21, max=201, step=2, description="θ pts")
    phi_bs_w = widgets.FloatSlider(value=np.pi/2, min=-np.pi, max=np.pi, step=0.01, description="φ_BS")

    # Scenario selection by ID (labels shown)
    scenario_w = widgets.SelectMultiple(
        options=[(SCENARIOS[sid]["label"], sid) for sid in SCENARIO_IDS],
        value=("fock11", "superpos01", "coherent", "smsv"),
        description="Scenarios",
        layout=widgets.Layout(width="420px", height="80px")
    )

    # De-emphasis selection by ID (None => no emphasis)
    deemph_w = widgets.Dropdown(
        options=[("(none)", None)] + [(SCENARIOS[sid]["label"], sid) for sid in SCENARIO_IDS],
        value=None,
        description="Emphasize"
    )

    # Coherent params
    alpha_abs_w = widgets.FloatSlider(value=1.0, min=0.0, max=3.0, step=0.01, description="|α|")
    alpha_phase_w = widgets.FloatSlider(value=0.0, min=-np.pi, max=np.pi, step=0.01, description="arg α")

    # SMSV params
    r_dB_w = widgets.FloatSlider(value=6.0, min=0.0, max=12.0, step=0.1, description="r (dB)")
    phi_sq1_w = widgets.FloatSlider(value=0.0, min=-np.pi, max=np.pi, step=0.01, description="φsq1")
    phi_sq2_w = widgets.FloatSlider(value=np.pi/2, min=-np.pi, max=np.pi, step=0.01, description="φsq2")

    # Spec tolerances
    delta_pc_w = widgets.FloatSlider(value=0.02, min=0.0, max=0.2, step=0.005, description="δPc")
    delta_nrf_w = widgets.FloatSlider(value=0.05, min=0.0, max=0.5, step=0.01, description="δNRF")

    run_btn = widgets.Button(description="Run sweep", button_style="primary")

    select_hint = widgets.HTML(
        value="""
        <div style="
            border:1px solid #bbb;
            border-radius:6px;
            padding:5px 7px;
            margin-top:6px;
            width:150px;
            box-sizing:border-box;
            font-size:11px;
            line-height:1.2;">
          Use Ctrl-left mouse click to select state families
        </div>
        """
    )

    # Outputs
    out = widgets.Output()
    cards = widgets.HBox(
        [],
        layout=widgets.Layout(
            gap="6px",
            justify_content="space-between",
            align_items="flex-start",
            width="100%"
        )
    )
    explain = widgets.Output()

    def render(_=None):
        with out:
            clear_output(wait=True)
        with explain:
            clear_output(wait=True)

        ids = list(scenario_w.value)
        if len(ids) == 0:
            with out:
                print("Select at least one scenario.")
            return
        if len(ids) > 4:
            with out:
                print("For no-scrolling layout, please select 4 or fewer scenarios.")
            return

        deemph_id = deemph_w.value  # None or scenario id

        theta_list, results_by_id, meta = run_hom_sweep(
            ids,
            N=int(N_w.value),
            theta_pts=int(theta_pts_w.value),
            phi_bs=float(phi_bs_w.value),
            alpha_abs=float(alpha_abs_w.value),
            alpha_phase=float(alpha_phase_w.value),
            r_dB=float(r_dB_w.value),
            phi_sq1=float(phi_sq1_w.value),
            phi_sq2=float(phi_sq2_w.value),
        )

        # ----- persist last run for Cell 3 -----
        globals()["HOM_LAST_META"] = meta
        globals()["HOM_LAST_IDS"] = ids
        globals()["HOM_LAST_DEEMPH_ID"] = deemph_id
        # --------------------------------------

        delta_pc = float(delta_pc_w.value)
        delta_nrf = float(delta_nrf_w.value)

        with out:
            print("HOM simulator (no loss): Pc, g2_12, NRF vs θ/π  (balanced at θ/π=0.5)")
            print(f"N_cut={meta['N_cut']}, θ_pts={meta['theta_pts']}, φ_BS={meta['phi_bs']:.2f}")
            print(f"Coherent: |α|={meta['alpha_abs']:.2f}, arg(α)={meta['alpha_phase']:+.2f}")
            print(f"SMSV: r={meta['r_dB']:.1f} dB, φ1={meta['phi_sq1']:+.2f}, φ2={meta['phi_sq2']:+.2f}, Δφ={meta['delta_phi_sq']:+.2f}")
            if deemph_id is None:
                print("Emphasis: (none)\n")
            else:
                print(f"Emphasis: {SCENARIOS[deemph_id]['label']} (id={deemph_id})\n")

            plot_three_panel(theta_list, results_by_id, meta, deemph_id=deemph_id)

        # Spec cards underneath (display labels include current parameters)
        html_cards = []
        for sid in ids:
            disp = make_display_label(sid, meta)
            s = compute_specs(theta_list, results_by_id[sid], delta_pc=delta_pc, delta_nrf=delta_nrf)
            html_cards.append(widgets.HTML(make_spec_card_html(disp, s, delta_pc, delta_nrf)))
        cards.children = tuple(html_cards)

        with explain:
            print("How to read the spec cards:")
            print("  • V_Pc: contrast from Pc(θ) sweep (larger = stronger dip relative to max).")
            print("  • tol_pc: half-width (θ/π) around 0.5 where Pc stays within δPc of Pc_min.")
            print("  • tol_nrf: half-width (θ/π) around 0.5 where NRF stays within δNRF of NRF_min.")
            print("Engineering take-away:")
            print("  No universally best state—selection depends on objective (dip contrast vs noise vs correlations).")

    run_btn.on_click(render)

    row1 = widgets.HBox([N_w, theta_pts_w, phi_bs_w, deemph_w])
    row2 = widgets.HBox([delta_pc_w, delta_nrf_w])
    row3 = widgets.HBox([scenario_w, widgets.VBox([run_btn, select_hint])])
    row4 = widgets.HBox([alpha_abs_w, alpha_phase_w])
    row5 = widgets.HBox([r_dB_w, phi_sq1_w, phi_sq2_w])

    return widgets.VBox([row1, row2, row3, row4, row5, out, cards, explain])

make_hom_ui_no_scroll_cards()